# SNN Temporal Credit Assignment — Scientific Validation v2

## What this notebook does

Trains and compares 4 learning rules on SNNs with a **scientifically valid** temporal task:

| Method | Expected Outcome |
|--------|------------------|
| **BPTT** | Highest accuracy (full backprop) |
| **TBPTT(K=10)** | Near-chance — K < delay, **genuinely fails** |
| **E-prop** | Partial learning via eligibility traces |
| **CTCA** | Better than E-prop, closer to BPTT |

**Task**: `delayed_xor` — 5-step cue → 50-step blank delay → 10-step readout window

**All 6 original bugs fixed** — see BUGS_FIXED.md for full report.

## Steps
1. Run Cell 1 to upload and extract the package
2. Run Cell 2 onwards — **do not skip Cell 4 (task validation)**


In [ ]:

import os, sys, subprocess

EXTRACT_DIR = "/content/"
ZIP_PATH    = None

try:
    from google.colab import files as _colab_files
    print("Colab detected. File picker will open — select snn_ctca_v2.zip")
    _uploaded = _colab_files.upload()
    _zip_keys = [k for k in _uploaded if k.endswith(".zip")]
    if _zip_keys:
        ZIP_PATH = _zip_keys[0]
        print(f"Uploaded: {ZIP_PATH}")
    else:
        raise RuntimeError("No .zip file uploaded. Please select snn_ctca_v2.zip")
except ImportError:
    for _p in ["/content/snn_ctca_v2.zip", "./snn_ctca_v2.zip",
                "/content/snn_ctca_package_fixed.zip", "./snn_ctca_package_fixed.zip"]:
        if os.path.exists(_p):
            ZIP_PATH = _p
            print(f"Found zip at: {ZIP_PATH}")
            break
    if ZIP_PATH is None:
        raise FileNotFoundError(
            "Cannot find snn_ctca_v2.zip.\n"
            "Upload it via the Colab Files panel or place it in the notebook directory."
        )

if not os.path.exists("/content/snn_ctca"):
    result = subprocess.run(["unzip", "-q", ZIP_PATH, "-d", EXTRACT_DIR],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Failed to extract: {result.stderr}")
    print("Extracted to /content/snn_ctca/")
else:
    print("/content/snn_ctca/ already exists")

for _p in ["/content/snn_ctca", "/content"]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

_contents = sorted(os.listdir("/content/snn_ctca"))
_required = ["configs.py", "models", "learning_rules", "experiments", "utils"]
_missing  = [r for r in _required if r not in _contents]
if _missing:
    raise RuntimeError(f"Extraction incomplete — missing: {_missing}")
print(f"Ready. Contents: {_contents}")


In [ ]:

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nWill train on: {DEVICE}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Import all modules
# ═══════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, '/content/snn_ctca')

from configs import (ExperimentConfig, SNNConfig, TaskConfig,
                     BPTTConfig, TruncBPTTConfig, EpropConfig, CTCAConfig)
from models.snn_model import build_model
from experiments.tasks import build_dataloaders, get_task_description, validate_task_scientifically
from experiments.run_comparison import sanity_check, run_full_comparison
from experiments.ablation import run_delay_sweep, run_tbptt_window_sweep
from utils.plotting import (plot_training_curves, plot_gradient_quality,
                             plot_final_comparison, plot_ablation_delay, plot_tbptt_window)
from utils.logging import print_comparison_table

import json
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display

%matplotlib inline
print('All imports successful')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Task Scientific Validation (MANDATORY)
# Verifies: no shortcut, no leakage, blank readout window
# ═══════════════════════════════════════════════════════════════
from configs import TaskConfig
task_cfg = TaskConfig()
print(f"Task: {get_task_description(task_cfg)}")
print()
checks = validate_task_scientifically(task_cfg, n=1000)
print("Scientific validity checks:")
for k, v in checks.items():
    if isinstance(v, bool):
        sym = '✅' if v else '❌'
        print(f"  {sym} {k} = {v}")
    else:
        print(f"     {k} = {v:.4f}")

assert checks['memoryless_ok'],  'FAIL: memoryless probe too accurate — shortcut possible!'
assert checks['no_leakage'],     'FAIL: label leakage in readout window!'
assert checks['readout_blank'],  'FAIL: readout window has signal!'
print("\nTask is scientifically valid. Safe to run experiment.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Sanity Check (~20 seconds)
# ═══════════════════════════════════════════════════════════════
from experiments.run_comparison import sanity_check
ok = sanity_check()
if not ok:
    raise RuntimeError('Sanity check FAILED — do not run training!')
print('\nSanity check passed. Ready to train.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Configure Full Experiment
# ═══════════════════════════════════════════════════════════════
cfg = ExperimentConfig()

# Model
cfg.snn.T           = 60
cfg.snn.hidden_dim  = 128
cfg.snn.n_layers    = 2
cfg.snn.readout_len = 10     # classify from last 10 timesteps ONLY
cfg.snn.tau_mem     = 20.0
cfg.snn.tau_syn     = 10.0
cfg.snn.v_th        = 0.5
cfg.snn.alpha_surr  = 1.0
cfg.snn.target_rate  = 0.10
cfg.snn.rate_penalty = 0.005

# Task: 5-step cue, 50-step delay, 10-step readout
cfg.task.T               = 60
cfg.task.cue_duration    = 5
cfg.task.delay           = 50
cfg.task.distractor_rate = 0.10
cfg.task.randomize_delay = False
cfg.task.n_train         = 4000
cfg.task.n_val           = 500
cfg.task.batch_size      = 32

# BPTT (reference)
cfg.bptt.epochs = 60;  cfg.bptt.lr = 5e-4

# TBPTT — K=10 << delay=50, must FAIL
cfg.tbptt.epochs = 60; cfg.tbptt.trunc_len = 10; cfg.tbptt.lr = 5e-4

# E-prop
cfg.eprop.epochs = 60; cfg.eprop.lr = 2e-3; cfg.eprop.tau_e = 30.0
cfg.eprop.alpha_surr = 1.0; cfg.eprop.head_lr_mult = 5.0

# CTCA
cfg.ctca.epochs = 60; cfg.ctca.lr = 2e-3; cfg.ctca.trunc_len = 60
cfg.ctca.influence_decay = 0.99; cfg.ctca.head_lr_mult = 5.0

cfg.methods    = ['bptt', 'tbptt', 'eprop', 'ctca']
cfg.log_every  = 10
cfg.output_dir = '/content/results'
cfg.device     = DEVICE
cfg.seed       = 42

n_params = build_model(cfg.snn).count_parameters()
print(f"Task  : {get_task_description(cfg.task)}")
print(f"Model : {n_params:,} params  ({cfg.snn.hidden_dim}h x {cfg.snn.n_layers}L)")
print(f"Device: {DEVICE}")
est = 60 * 4 * (5 if DEVICE == 'cuda' else 25)
print(f"Est.  : ~{est} min on {DEVICE}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Run Full Experiment
# ═══════════════════════════════════════════════════════════════
import time
t0 = time.time()
results = run_full_comparison(cfg)
print(f'\nDone in {(time.time()-t0)/60:.1f} min')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — Load + Display Results
# ═══════════════════════════════════════════════════════════════
task    = cfg.task.task_name
out_dir = Path(cfg.output_dir)
with open(out_dir / f'history_{task}.json') as f:   histories = json.load(f)
with open(out_dir / f'results_{task}.json') as f:   summaries = json.load(f)

print('Best validation accuracy per method:')
for m, s in summaries.items():
    acc  = s.get('best_val_acc', 0.0)
    rate = s.get('mean_spike_rate', 0.0)
    cos  = s.get('max_grad_cosine', float('nan'))
    bar  = '\u2588' * int(acc * 30)
    print(f'  {m:8s}  acc={acc:.3f}  rate={rate:.3f}  cos={cos:.3f}  {bar}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — Scientific Validity Assert
# ═══════════════════════════════════════════════════════════════
bptt_acc  = summaries['bptt']['best_val_acc']
tbptt_acc = summaries['tbptt']['best_val_acc']
ctca_acc  = summaries['ctca']['best_val_acc']
eprop_acc = summaries['eprop']['best_val_acc']
ctca_cos  = summaries['ctca'].get('max_grad_cosine', float('nan'))
eprop_cos = summaries['eprop'].get('max_grad_cosine', float('nan'))

scientific_checks = [
    ('BPTT learns (>0.70)',          bptt_acc  > 0.70),
    ('TBPTT fails (<0.70)',          tbptt_acc < 0.70),
    ('BPTT > TBPTT by >0.15',        bptt_acc  > tbptt_acc + 0.15),
    ('CTCA cos > E-prop cos',        ctca_cos  > eprop_cos),
    ('E-prop learns (>0.55)',         eprop_acc > 0.55),
    ('CTCA learns (>0.60)',          ctca_acc  > 0.60),
    ('No all-method tie at 1.0',     not all(v > 0.99 for v in [bptt_acc, tbptt_acc, eprop_acc, ctca_acc])),
]
all_ok = True
for name, cond in scientific_checks:
    sym = '\u2705' if cond else '\u274c'
    print(f'  {sym} {name}')
    all_ok = all_ok and cond

print(f'\nVERDICT: {"SCIENTIFICALLY VALID" if all_ok else "INVALID — debug before publishing"}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 10 — Plots
# ═══════════════════════════════════════════════════════════════
fig1 = plot_training_curves(histories, task, save_path=str(out_dir/f'curves_{task}.png'))
display(fig1); plt.close(fig1)

fig2 = plot_gradient_quality(histories, task, save_path=str(out_dir/f'gradients_{task}.png'))
display(fig2); plt.close(fig2)

fig3 = plot_final_comparison(summaries, save_path=str(out_dir/f'comparison_{task}.png'))
display(fig3); plt.close(fig3)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 11 — Ablation: TBPTT Window Sweep
# Key test: TBPTT(K < delay) must fail; TBPTT(K >= delay) must learn
# ═══════════════════════════════════════════════════════════════
ablation_tbptt = run_tbptt_window_sweep(Ks=[5, 10, 20, 50], delay=30)
fig4 = plot_tbptt_window(ablation_tbptt, delay=30,
                          save_path=str(out_dir/'ablation_tbptt_window.png'))
display(fig4); plt.close(fig4)

print('TBPTT window sweep (should fail when K < 30):')
for K, r in sorted(ablation_tbptt.items()):
    verdict = 'FAILS (expected)' if K < 30 and r['tbptt']['acc'] < 0.65 else \
              'LEARNS (expected)' if K >= 30 else 'UNEXPECTED'
    print(f'  K={K:3d}: tbptt={r["tbptt"]["acc"]:.3f}  [{verdict}]')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 12 — Ablation: Delay Sweep
# Performance should degrade with longer delays for all methods
# ═══════════════════════════════════════════════════════════════
ablation_delay = run_delay_sweep(delays=[10, 20, 30, 50])
fig5 = plot_ablation_delay(ablation_delay, save_path=str(out_dir/'ablation_delay.png'))
display(fig5); plt.close(fig5)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 13 — Final Summary Table
# ═══════════════════════════════════════════════════════════════
print_comparison_table(summaries)
print()
print('Expected outcomes (scientifically valid experiment):')
print('  BPTT      : >85%  — full temporal backprop')
print('  CTCA      : 65-80% — W^T sweep, correct gradient direction')
print('  E-prop    : 55-70% — random feedback, approximate')
print('  TBPTT(K=10): ~50%  — cannot bridge 50-step delay')
print()
print('Gradient cosine: CTCA > E-prop (theoretically guaranteed)')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 14 — Download Results
# ═══════════════════════════════════════════════════════════════
import shutil
archive = '/content/snn_results_v2'
shutil.make_archive(archive, 'zip', '/content/results')
print(f'Archive: {archive}.zip')
try:
    from google.colab import files
    files.download(f'{archive}.zip')
    print('Download started.')
except ImportError:
    print(f'Not in Colab — find results at {archive}.zip')
